# 02 · Limpieza de la base de Dengue

Entrada: `datos-procesados/dengue_merge.csv.gz`, generado en `01_merge_datos.ipynb` (1,276,897 registros únicos, 2020–2026).
Salida: `datos-procesados/dengue_limpio.csv`, una fila por paciente **confirmado** con las variables elegidas para la regresión logística.

| Paso | Quita | % quitado | Quedan | Muertes |
|---|---|---|---|---|
| 0 · Base del merge | – | – | 1,276,897 | 5,296 |
| 1 · Solo registros en la foto final | 22,629 | 1.77% | 1,254,268 | 5,207 |
| 2 · Solo confirmados (`ESTATUS_CASO = 2`) | 1,002,367 | 79.92% | 251,901 | 1,763 |
| 3 · Quitar entidad de asignación no especificada (`ENTIDAD_ASIG = 99`) | 1 | 0.0004% | 251,900 | 1,763 |
| **Total** | **1,024,997** | **80.27%** | **251,900** | **1,763 (0.70%)** |

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("datos-procesados")
dengue = pd.read_csv(DATA / "dengue_merge.csv.gz",
                     parse_dates=["FECHA_SIGN_SINTOMAS", "FECHA_ACTUALIZACION"])
N0 = len(dengue)

def reporte(paso, antes, despues):
    q = len(antes) - len(despues)
    print(f"{paso}: quita {q:,} de {len(antes):,} filas ({q / len(antes):.2%} del paso, "
          f"{q / N0:.2%} de la base original) → quedan {len(despues):,} | "
          f"muertes: {(despues.DEFUNCION == 1).sum():,}")

print(f"Base original: {N0:,} filas | muertes: {(dengue.DEFUNCION == 1).sum():,}")

Base original: 1,276,897 filas | muertes: 5,296


## Paso 1 · Solo registros que siguen en la foto final
Se quitan los registros que aparecieron en alguna foto semanal pero ya no están en la **última foto de su año** (`EN_FOTO_FINAL = False`). Son registros que la DGE dio de baja.

**Quita 22,629 de 1,276,897 filas (1.77%).** Entre los confirmados solo son 1,266 (0.50%) y 7 muertes.

### ¿Y los registros donde cambió `DEFUNCION`?
**Decisión del equipo: no se quitan; solo se filtra por foto final** (`QUITAR_DEFUNCION_CAMBIO = False`). De 2,494 cambios, **2,433 son de "No" → "Sí"**: la muerte se registró después. Su valor final es el correcto. Quitarlos eliminaría **801 de 1,770 muertes confirmadas (45%)**, y el modelo solo vería las muertes que se reportaron rápido (sesgo de selección).

Si se cambia a `True`: quita 2,476 filas más (0.20%) y al final quedan **251,077 confirmados con solo 962 muertes**.

In [2]:
QUITAR_DEFUNCION_CAMBIO = False

antes = dengue
dengue = dengue[dengue.EN_FOTO_FINAL]
reporte("Paso 1a · foto final", antes, dengue)

if QUITAR_DEFUNCION_CAMBIO:
    antes = dengue
    dengue = dengue[~dengue.DEFUNCION_CAMBIO]
    reporte("Paso 1b · DEFUNCION sin cambios", antes, dengue)

Paso 1a · foto final: quita 22,629 de 1,276,897 filas (1.77% del paso, 1.77% de la base original) → quedan 1,254,268 | muertes: 5,207


## Paso 2 · Solo casos confirmados de dengue
Catálogo `ESTATUS_CASO`: 1 = Probable, 2 = Confirmado, 3 = Descartado. Se queda solo el **2**. Después la columna se elimina porque ya es constante.

Base original: 738,892 probables (57.9%), 253,167 confirmados (19.8%) y 284,838 descartados (22.3%).

**Quita 1,002,367 de 1,254,268 filas (79.92%).** Quedan **251,901 confirmados con 1,763 muertes (0.70%)**.

In [3]:
print(dengue.ESTATUS_CASO.map({1: "Probable", 2: "Confirmado", 3: "Descartado"}).value_counts())

antes = dengue
dengue = dengue[dengue.ESTATUS_CASO == 2].drop(columns="ESTATUS_CASO")
reporte("Paso 2 · confirmados", antes, dengue)

ESTATUS_CASO
Probable      719278
Descartado    283089
Confirmado    251901
Name: count, dtype: int64


Paso 2 · confirmados: quita 1,002,367 de 1,254,268 filas (79.92% del paso, 78.50% de la base original) → quedan 251,901 | muertes: 1,763


## Paso 3 · Quitar `ENTIDAD_ASIG = 99`
La clave 99 indica que no se especificó la entidad de asignación. **Quita 1 de 251,901 filas (0.0004%)**: un sobreviviente que reside y se notificó en el Estado de México (15).

In [4]:
antes = dengue
dengue = dengue[dengue.ENTIDAD_ASIG != 99]
reporte("Paso 3 · ENTIDAD_ASIG != 99", antes, dengue)

Paso 3 · ENTIDAD_ASIG != 99: quita 1 de 251,901 filas (0.00% del paso, 0.00% de la base original) → quedan 251,900 | muertes: 1,763


## Paso 4 · Selección de variables
Variables elegidas (archivo `VARIABLES .rtf`):

| Variable | Columna | Tipo |
|---|---|---|
| **Objetivo** | `DEFUNCION` | binaria |
| Sexo | `SEXO` → `HOMBRE` | binaria |
| Edad | `EDAD_ANOS` | numérica |
| Mes de inicio de síntomas | `FECHA_SIGN_SINTOMAS` → `MES_ENE` … `MES_DIC` | dummies (referencia: octubre) |
| Entidad de asignación | `ENTIDAD_ASIG` → `COSTERA` | binaria (1 = estado costero) |
| Indígena, Hemorrágicos, Diabetes, Hipertensión, Enf. renal, Inmunosupresión, Cirrosis hepática, Embarazo | `INDIGENA` … `EMBARAZO` | binarias |
| Tipo de paciente | `TIPO_PACIENTE` → `HOSPITALIZADO` | binaria |
| Resultado PCR (serotipo) | `RESULTADO_PCR` → `DENV2`, `DENV3`, `DENV4`, `SIN_SEROTIPO` | dummies (referencia: DENV1) |

Se conserva `ID_REGISTRO` como identificador. **No** entran al modelo: dictamen, úlcera péptica, habla lengua indígena, toma de muestra, municipios, entidad de residencia, institución y fechas de actualización.

⚠️ **Notas para el modelado:**
- `DICTAMEN` se excluye: solo se asigna a quienes **murieron** (entre los confirmados, 250,135 de los 250,138 vivos tienen "No aplica"). Es consecuencia del desenlace y produciría una separación casi perfecta (fuga de información).
- "Conteo o porcentaje de muertos por año" no se calcula aquí, porque derivarlo del mismo `DEFUNCION` también es fuga de información. Además, el equipo decidió **no incluir el año** (ver Paso 5c).

In [5]:
VARIABLES = [
    "ID_REGISTRO",
    "DEFUNCION",
    "SEXO", "EDAD_ANOS", "FECHA_SIGN_SINTOMAS", "ANIO_SINTOMAS", "ENTIDAD_ASIG",
    "INDIGENA", "HEMORRAGICOS", "DIABETES", "HIPERTENSION",
    "ENFERMEDAD_RENAL", "INMUNOSUPR", "CIRROSIS_HEPATICA", "EMBARAZO",
    "TIPO_PACIENTE", "RESULTADO_PCR",
]
n_cols = dengue.shape[1]
dengue = dengue[VARIABLES].copy()
print(f"Columnas: {n_cols} → {dengue.shape[1]}")

Columnas: 35 → 17


## Paso 5 · Recodificación a 1/0
El catálogo SI_NO usa **1 = Sí, 2 = No**. Para la regresión logística se pasa a **1 = Sí, 0 = No**.
- `DEFUNCION`: 1 = murió, 0 = no murió.
- `SEXO` (1 = Mujer, 2 = Hombre) → `HOMBRE`: 1 = hombre, 0 = mujer.
- `TIPO_PACIENTE` (1 = Ambulatorio, 2 = Hospitalizado) → `HOSPITALIZADO`: 1 = hospitalizado, 0 = ambulatorio. **9 registros** (0.004%, todos sobrevivientes de 2021) no tienen tipo de paciente y **se asignan como ambulatorios (0)**.
- `ENTIDAD_ASIG` → `COSTERA`: 1 = la entidad de asignación tiene costa, 0 = no tiene. Son 17 entidades costeras: Baja California (02), Baja California Sur (03), Sonora (26), Sinaloa (25), Nayarit (18), Jalisco (14), Colima (06), Michoacán (16), Guerrero (12), Oaxaca (20), Chiapas (07), Tamaulipas (28), Veracruz (30), Tabasco (27), Campeche (04), Yucatán (31) y Quintana Roo (23). Las claves se toman del catálogo `CATÁLOGO ENTIDAD`. Después se elimina `ENTIDAD_ASIG`.
- `EMBARAZO` vale 0 en todos los hombres.
- `RESULTADO_PCR` se convierte en **variables dummy** (ver Paso 5b).

Ninguna variable binaria tiene valores fuera de {1, 2}, así que la recodificación no quita filas (**0%**).

In [6]:
BINARIAS = ["DEFUNCION", "INDIGENA", "HEMORRAGICOS", "DIABETES", "HIPERTENSION",
            "ENFERMEDAD_RENAL", "INMUNOSUPR", "CIRROSIS_HEPATICA", "EMBARAZO"]

for col in BINARIAS + ["SEXO"]:
    assert set(dengue[col].unique()) <= {1, 2}, f"{col}: {dengue[col].unique()}"

for col in BINARIAS:
    dengue[col] = (dengue[col] == 1).astype("int8")

dengue["HOMBRE"] = (dengue.SEXO == 2).astype("int8")
dengue = dengue.drop(columns="SEXO")

# Tipo de paciente: vacíos → ambulatorio (0); hospitalizado = 1
assert set(dengue.TIPO_PACIENTE.dropna().unique()) <= {1, 2}
print(f"TIPO_PACIENTE vacíos asignados a ambulatorio: {dengue.TIPO_PACIENTE.isna().sum()}")
dengue["HOSPITALIZADO"] = (dengue.TIPO_PACIENTE == 2).astype("int8")
dengue = dengue.drop(columns="TIPO_PACIENTE")

# Entidad de asignación → COSTERA (claves del catálogo oficial)
entidades = pd.read_excel("datos-abiertos/diccionario/Catalogos_Dengue.xlsx", sheet_name="CATÁLOGO ENTIDAD")
COSTERAS = {"BC", "BS", "SR", "SL", "NT", "JC", "CM", "MN", "GR", "OC", "CS", "TS", "VZ", "TC", "CC", "YN", "QR"}
claves_costeras = set(entidades.loc[entidades.ABREVIATURA.str.strip().isin(COSTERAS), "CLAVE_ENTIDAD"])
assert len(claves_costeras) == 17
print("Entidades costeras:", entidades.loc[entidades.CLAVE_ENTIDAD.isin(claves_costeras), "ENTIDAD_FEDERATIVA"].tolist())

assert dengue.ENTIDAD_ASIG.between(1, 32).all()
dengue["COSTERA"] = dengue.ENTIDAD_ASIG.isin(claves_costeras).astype("int8")
dengue = dengue.drop(columns="ENTIDAD_ASIG")

dengue[BINARIAS + ["HOMBRE", "HOSPITALIZADO", "COSTERA"]].agg(["sum", "mean"]).T.rename(columns={"sum": "n_si", "mean": "proporcion"})

TIPO_PACIENTE vacíos asignados a ambulatorio: 9


Entidades costeras: ['BAJA CALIFORNIA', 'BAJA CALIFORNIA SUR', 'CAMPECHE', 'COLIMA', 'CHIAPAS', 'GUERRERO', 'JALISCO', 'MICHOACÁN DE OCAMPO', 'NAYARIT', 'OAXACA', 'QUINTANA ROO', 'SINALOA', 'SONORA', 'TABASCO', 'TAMAULIPAS', 'VERACRUZ DE IGNACIO DE LA LLAVE', 'YUCATÁN']


,n_si,proporcion
DEFUNCION,1763.0,0.006999
INDIGENA,8348.0,0.033140
HEMORRAGICOS,2287.0,0.009079
DIABETES,7253.0,0.028793
HIPERTENSION,5918.0,0.023493
ENFERMEDAD_RENAL,633.0,0.002513
INMUNOSUPR,371.0,0.001473
CIRROSIS_HEPATICA,233.0,0.000925
EMBARAZO,6602.0,0.026209
HOMBRE,112788.0,0.447749


## Paso 5b · Serotipo (`RESULTADO_PCR`) → variables dummy
Catálogo: 1 = DENV1, 2 = DENV2, 3 = DENV3, 4 = DENV4, 5 = Sin serotipo aislado.

`RESULTADO_PCR` tiene 5 categorías y se convierte en **4 columnas 1/0**. **DENV1 es la categoría de referencia**: un paciente con las 4 dummies en 0 es DENV1. Si se incluyeran las 5 columnas, siempre sumarían 1 y serían colineales con el intercepto (*trampa de las dummies*).

En el modelo, cada coeficiente se interpreta **en relación con DENV1**, manteniendo fijas las demás variables.

| Serotipo | Casos | % | Muertes | Mortalidad |
|---|---|---|---|---|
| DENV1 (referencia) | 13,531 | 5.4% | 39 | 0.29% |
| DENV2 | 23,139 | 9.2% | 153 | 0.66% |
| DENV3 | 71,812 | 28.5% | 479 | 0.67% |
| DENV4 | 1,396 | 0.6% | 12 | 0.86% |
| Sin serotipo | 142,022 | 56.4% | 1,080 | 0.76% |

**Sin serotipo** (56.4% de los confirmados): son casos confirmados de dengue en los que el PCR no identificó el serotipo. Las causas probables son: la confirmación fue por NS1 o IgM (no todas las muestras se procesan por PCR); la muestra se tomó después de ~5 días de síntomas, cuando el virus ya no se detecta en sangre; o el caso se confirmó por asociación epidemiológica sin muestra (562 casos). Es un serotipo **desconocido**, no un quinto virus.

No quita filas (**0%**).

In [7]:
SEROTIPOS = {1: "DENV1", 2: "DENV2", 3: "DENV3", 4: "DENV4", 5: "SIN_SEROTIPO"}
assert set(dengue.RESULTADO_PCR.unique()) <= set(SEROTIPOS)

serotipo = pd.Categorical(dengue.RESULTADO_PCR.map(SEROTIPOS), categories=list(SEROTIPOS.values()))
dummies = pd.get_dummies(serotipo, drop_first=True, dtype="int8")  # drop_first quita DENV1 (referencia)
dummies.index = dengue.index
dengue = pd.concat([dengue.drop(columns="RESULTADO_PCR"), dummies], axis=1)

print("Dummies creadas:", list(dummies.columns), "| referencia: DENV1")
assert (dummies.sum(axis=1) <= 1).all()

tabla = (pd.Series(serotipo, index=dengue.index).to_frame("serotipo").assign(DEFUNCION=dengue.DEFUNCION)
         .groupby("serotipo", observed=True).DEFUNCION.agg(casos="size", muertes="sum"))
tabla["% casos"] = (tabla.casos / tabla.casos.sum()).map("{:.1%}".format)
tabla["mortalidad"] = (tabla.muertes / tabla.casos).map("{:.2%}".format)
tabla

Dummies creadas: ['DENV2', 'DENV3', 'DENV4', 'SIN_SEROTIPO'] | referencia: DENV1


,casos,muertes,% casos,mortalidad
serotipo,,,,
DENV1,13531,39,5.4%,0.29%
DENV2,23139,153,9.2%,0.66%
DENV3,71812,479,28.5%,0.67%
DENV4,1396,12,0.6%,0.86%
SIN_SEROTIPO,142022,1080,56.4%,0.76%


## Paso 5c · Mes de inicio de síntomas → variables dummy (se quita el año)
**Año: se excluye.** La mortalidad por año no muestra una tendencia clara (0.58%–0.86%), y los avances médicos entre años no implican necesariamente menos muertes. *Limitación:* el serotipo dominante cambió con los años (DENV2 en 2020–2021 y DENV3 desde 2023), así que sin el año los coeficientes de serotipo absorben parte del efecto de la época.

| Año | 2020 | 2021 | 2022 | 2023 | 2024 | 2025 | 2026 |
|---|---|---|---|---|---|---|---|
| Confirmados | 24,308 | 6,553 | 12,327 | 54,197 | 123,141 | 21,918 | 9,456 |
| Mortalidad | 0.58% | 0.76% | 0.66% | 0.73% | 0.69% | 0.86% | 0.58% |

**Mes: se incluye para controlar la estacionalidad.** En temporada de lluvias (junio–octubre) se concentra el 72% de los casos, pero su mortalidad es **menor** (0.68% contra 0.76% en secas). En temporada alta se confirman muchos casos leves. En secas se estudia sobre todo a los graves: la hospitalización llega a ~48% en enero–marzo. El mes capta ese cambio en el perfil de los casos confirmados.

Se crean **11 dummies**. **Octubre es la referencia** porque es el mes con más casos (21.7%), lo que da coeficientes más estables. Cada coeficiente se lee como "iniciar síntomas en ese mes, comparado con octubre". El mes con menos muertes (diciembre) tiene 31, suficientes para estimarlo.

Después se eliminan `FECHA_SIGN_SINTOMAS` y `ANIO_SINTOMAS`. No quita filas (**0%**).

In [8]:
MESES = {1: "MES_ENE", 2: "MES_FEB", 3: "MES_MAR", 4: "MES_ABR", 5: "MES_MAY", 6: "MES_JUN",
         7: "MES_JUL", 8: "MES_AGO", 9: "MES_SEP", 10: "MES_OCT", 11: "MES_NOV", 12: "MES_DIC"}
REF_MES = "MES_OCT"

assert dengue.FECHA_SIGN_SINTOMAS.notna().all()
mes = dengue.FECHA_SIGN_SINTOMAS.dt.month.map(MESES)

# Descriptivo por mes (antes de convertir)
desc = dengue.assign(MES=pd.Categorical(mes, categories=list(MESES.values()))).groupby("MES", observed=True).agg(
    casos=("DEFUNCION", "size"), muertes=("DEFUNCION", "sum"), hosp=("HOSPITALIZADO", "mean"))
desc["% casos"] = (desc.casos / desc.casos.sum()).map("{:.1%}".format)
desc["mortalidad"] = (desc.muertes / desc.casos).map("{:.2%}".format)
desc["hospitalizados"] = desc.hosp.map("{:.1%}".format)
display(desc.drop(columns="hosp"))

dummies_mes = pd.get_dummies(mes, dtype="int8").drop(columns=REF_MES)
dummies_mes = dummies_mes[[m for m in MESES.values() if m != REF_MES]]
assert (dummies_mes.sum(axis=1) <= 1).all()

dengue = pd.concat([dengue.drop(columns=["FECHA_SIGN_SINTOMAS", "ANIO_SINTOMAS"]), dummies_mes], axis=1)
print("Dummies creadas:", list(dummies_mes.columns), "| referencia:", REF_MES)

,casos,muertes,% casos,mortalidad,hospitalizados
MES,,,,,
MES_ENE,9194,76,3.6%,0.83%,49.2%
MES_FEB,5456,44,2.2%,0.81%,48.7%
MES_MAR,4974,38,2.0%,0.76%,47.8%
MES_ABR,5029,51,2.0%,1.01%,44.3%
MES_MAY,7005,59,2.8%,0.84%,41.3%
MES_JUN,13110,102,5.2%,0.78%,35.0%
MES_JUL,20056,142,8.0%,0.71%,36.7%
MES_AGO,39709,260,15.8%,0.65%,36.2%
MES_SEP,53148,349,21.1%,0.66%,37.2%


Dummies creadas: ['MES_ENE', 'MES_FEB', 'MES_MAR', 'MES_ABR', 'MES_MAY', 'MES_JUN', 'MES_JUL', 'MES_AGO', 'MES_SEP', 'MES_NOV', 'MES_DIC'] | referencia: MES_OCT


## Paso 6 · Revisión de valores faltantes
Solo para revisar. Ya no debe quedar ningún faltante.

In [9]:
print("Faltantes por columna:", dengue.isna().sum()[lambda s: s > 0].to_dict())
assert dengue.notna().all().all()
print("Edad:", dengue.EDAD_ANOS.describe().round(1).to_dict())

Faltantes por columna: {}
Edad: {'count': 251900.0, 'mean': 28.5, 'std': 17.8, 'min': 0.0, '25%': 14.0, '50%': 25.0, '75%': 40.0, 'max': 112.0}


## Paso 7 · Guardado

In [10]:
orden = ["ID_REGISTRO", "DEFUNCION", "HOMBRE", "EDAD_ANOS",
         "COSTERA", "INDIGENA", "HEMORRAGICOS", "DIABETES", "HIPERTENSION",
         "ENFERMEDAD_RENAL", "INMUNOSUPR", "CIRROSIS_HEPATICA", "EMBARAZO", "HOSPITALIZADO",
         "DENV2", "DENV3", "DENV4", "SIN_SEROTIPO"] + [m for m in MESES.values() if m != REF_MES]
dengue = dengue[orden].reset_index(drop=True)
dengue.to_csv(DATA / "dengue_limpio.csv", index=False)

print(f"Final: {dengue.shape[0]:,} filas × {dengue.shape[1]} columnas "
      f"({1 - len(dengue) / N0:.2%} de la base original quitado)")
print(f"{(DATA / 'dengue_limpio.csv').stat().st_size / 1e6:.1f} MB")
dengue.head()

Final: 251,900 filas × 29 columnas (80.27% de la base original quitado)
16.3 MB


,ID_REGISTRO,DEFUNCION,HOMBRE,EDAD_ANOS,COSTERA,INDIGENA,HEMORRAGICOS,DIABETES,HIPERTENSION,ENFERMEDAD_RENAL,...,MES_FEB,MES_MAR,MES_ABR,MES_MAY,MES_JUN,MES_JUL,MES_AGO,MES_SEP,MES_NOV,MES_DIC
0,759431,1,1,22,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,759683,0,1,36,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,759744,0,0,3,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,759961,0,1,22,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,760071,0,0,8,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
